In [20]:
# Target-horizon matrix
# Compare predictive performance across:
#   Targets: return, realised volatility, direction
#   Horizons: 1, 5, 10, 21 trading days
#
# Predictors at time t will only be used to predict outcomes after time t.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

# Regression
from sklearn.linear_model import Lasso, ElasticNet, Ridge
from sklearn.metrics import mean_squared_error, r2_score

# Classification
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

HORIZONS = [1, 5, 10, 21]

print("Target horizons:", HORIZONS)
print("Target families: return, realised volatility, direction")

Target horizons: [1, 5, 10, 21]
Target families: return, realised volatility, direction


In [2]:
# Load final DeepSeek and Gemma modelling datasets

DATA_DIR = Path("../data/processed")

deepseek = pd.read_csv(
    DATA_DIR / "macro_geopolitical_all_deepseek.csv",
    parse_dates=["observation_date"]
)

gemma = pd.read_csv(
    DATA_DIR / "macro_geopolitical_all_gemma.csv",
    parse_dates=["observation_date"]
)

print("DeepSeek shape:", deepseek.shape)
print("Gemma shape:", gemma.shape)

print("\nDeepSeek dates:")
print(deepseek["observation_date"].min(), "to", deepseek["observation_date"].max())

print("\nGemma dates:")
print(gemma["observation_date"].min(), "to", gemma["observation_date"].max())

DeepSeek shape: (993, 20)
Gemma shape: (993, 20)

DeepSeek dates:
2017-01-10 00:00:00 to 2021-01-08 00:00:00

Gemma dates:
2017-01-10 00:00:00 to 2021-01-08 00:00:00


In [3]:
# Inspect columns and verify base alignment

print("DEEPSEEK COLUMNS:")
print(deepseek.columns.tolist())

print("\nGEMMA COLUMNS:")
print(gemma.columns.tolist())

# Identify columns common to both datasets
common_cols = [
    col for col in deepseek.columns
    if col in gemma.columns
]

print("\nNumber of common columns:", len(common_cols))
print("Common columns:")
print(common_cols)

# Check dates are exactly identical row-by-row
dates_identical = deepseek["observation_date"].equals(
    gemma["observation_date"]
)

print("\nDates identical row-by-row:", dates_identical)

DEEPSEEK COLUMNS:
['observation_date', 'DEXUSEU_logreturn', 'DGS2_diff', 'USEPUINDXD_diff', 'VIXCLS', 'trade_sum', 'sanctions_sum', 'fed_pressure_sum', 'trade_max', 'sanctions_max', 'fed_pressure_max', 'trade_sum_ma3', 'sanctions_sum_ma3', 'fed_pressure_sum_ma3', 'trade_sum_ma5', 'sanctions_sum_ma5', 'fed_pressure_sum_ma5', 'trade_sum_ma10', 'sanctions_sum_ma10', 'fed_pressure_sum_ma10']

GEMMA COLUMNS:
['observation_date', 'DEXUSEU_logreturn', 'DGS2_diff', 'USEPUINDXD_diff', 'VIXCLS', 'trade_sum', 'sanctions_sum', 'fed_pressure_sum', 'trade_max', 'sanctions_max', 'fed_pressure_max', 'trade_sum_ma3', 'sanctions_sum_ma3', 'fed_pressure_sum_ma3', 'trade_sum_ma5', 'sanctions_sum_ma5', 'fed_pressure_sum_ma5', 'trade_sum_ma10', 'sanctions_sum_ma10', 'fed_pressure_sum_ma10']

Number of common columns: 20
Common columns:
['observation_date', 'DEXUSEU_logreturn', 'DGS2_diff', 'USEPUINDXD_diff', 'VIXCLS', 'trade_sum', 'sanctions_sum', 'fed_pressure_sum', 'trade_max', 'sanctions_max', 'fed_press

In [4]:
# Construct forward targets for all horizons
#
# At date t:
#   return_h    = cumulative log return from t+1 to t+h
#   volatility_h = realised volatility from t+1 to t+h
#   direction_h  = 1 if cumulative forward return > 0, else 0
#
# No contemporaneous return at t is included in the target.

HORIZONS = [1, 5, 10, 21]

r = deepseek["DEXUSEU_logreturn"]

targets = pd.DataFrame({
    "observation_date": deepseek["observation_date"]
})

for h in HORIZONS:

    # Future daily returns: t+1, ..., t+h
    future_returns = pd.concat(
        [r.shift(-j) for j in range(1, h + 1)],
        axis=1
    )

    # h-day cumulative forward log return
    targets[f"return_{h}d"] = future_returns.sum(
        axis=1,
        min_count=h
    )

    # h-day realised volatility
    targets[f"volatility_{h}d"] = np.sqrt(
        future_returns.pow(2).sum(
            axis=1,
            min_count=h
        )
    )

    # Direction of the h-day cumulative return
    targets[f"direction_{h}d"] = np.where(
        targets[f"return_{h}d"].notna(),
        (targets[f"return_{h}d"] > 0).astype(int),
        np.nan
    )

print("Target matrix shape:", targets.shape)

display(targets.head())

print("\nMissing values by target:")
print(targets.isna().sum())

Target matrix shape: (993, 13)


,observation_date,return_1d,volatility_1d,direction_1d,return_5d,volatility_5d,direction_5d,return_10d,volatility_10d,direction_10d,return_21d,volatility_21d,direction_21d
0,2017-01-10,-0.006739,0.006739,0.0,0.010351,0.018652,1.0,0.009227,0.022917,1.0,0.007351,0.027220,1.0
1,2017-01-11,0.015591,0.015591,1.0,0.012210,0.018064,1.0,0.017838,0.021984,1.0,0.009666,0.026741,1.0
2,2017-01-12,-0.003851,0.003851,0.0,0.006914,0.013756,1.0,0.001405,0.015523,1.0,-0.008379,0.021865,0.0
3,2017-01-13,0.006567,0.006567,1.0,0.011603,0.013233,1.0,0.015781,0.018354,1.0,-0.002639,0.021605,0.0
4,2017-01-17,-0.001216,0.001216,0.0,0.004478,0.011502,1.0,0.005873,0.017462,1.0,-0.003278,0.021420,0.0



Missing values by target:
observation_date     0
return_1d            1
volatility_1d        1
direction_1d         1
return_5d            5
volatility_5d        5
direction_5d         5
return_10d          10
volatility_10d      10
direction_10d       10
return_21d          21
volatility_21d      21
direction_21d       21
dtype: int64


In [5]:
# Manual audit of forward target construction

AUDIT_ROW = 100

audit_date = targets.loc[AUDIT_ROW, "observation_date"]

print("Audit date:", audit_date)
print("Return at t (must NOT enter any target):",
      deepseek.loc[AUDIT_ROW, "DEXUSEU_logreturn"])

for h in HORIZONS:

    future_r = deepseek.loc[
        AUDIT_ROW + 1 : AUDIT_ROW + h,
        "DEXUSEU_logreturn"
    ].to_numpy()

    manual_return = future_r.sum()
    manual_volatility = np.sqrt(np.sum(future_r ** 2))
    manual_direction = int(manual_return > 0)

    stored_return = targets.loc[AUDIT_ROW, f"return_{h}d"]
    stored_volatility = targets.loc[AUDIT_ROW, f"volatility_{h}d"]
    stored_direction = targets.loc[AUDIT_ROW, f"direction_{h}d"]

    print(f"\n--- {h}-DAY TARGET ---")
    print("Future returns used:", future_r)

    print("Manual return:       ", manual_return)
    print("Stored return:       ", stored_return)
    print("Return matches:      ",
          np.isclose(manual_return, stored_return))

    print("Manual volatility:   ", manual_volatility)
    print("Stored volatility:   ", stored_volatility)
    print("Volatility matches:  ",
          np.isclose(manual_volatility, stored_volatility))

    print("Manual direction:    ", manual_direction)
    print("Stored direction:    ", stored_direction)
    print("Direction matches:   ",
          manual_direction == stored_direction)

Audit date: 2017-06-06 00:00:00
Return at t (must NOT enter any target): 0.0014212118220931

--- 1-DAY TARGET ---
Future returns used: [-0.00266643]
Manual return:        -0.002666431230525
Stored return:        -0.002666431230525
Return matches:       True
Manual volatility:    0.002666431230525
Stored volatility:    0.002666431230525
Volatility matches:   True
Manual direction:     0
Stored direction:     0.0
Direction matches:    True

--- 5-DAY TARGET ---
Future returns used: [-0.00266643 -0.00169242 -0.00240996  0.00125034 -0.00089294]
Manual return:        -0.0064114200033543
Stored return:        -0.0064114200033543
Return matches:       True
Manual volatility:    0.004259430606901171
Stored volatility:    0.004259430606901171
Volatility matches:   True
Manual direction:     0
Stored direction:     0.0
Direction matches:    True

--- 10-DAY TARGET ---
Future returns used: [-0.00266643 -0.00169242 -0.00240996  0.00125034 -0.00089294  0.00738733
 -0.0111464   0.00375907 -0.0030419

In [6]:
# Inspect current DeepSeek and Gemma geopolitical features

macro_cols = [
    "DEXUSEU_logreturn",
    "DGS2_diff",
    "USEPUINDXD_diff",
    "VIXCLS"
]

deepseek_geop_cols = [
    col for col in deepseek.columns
    if col not in ["observation_date"] + macro_cols
]

gemma_geop_cols = [
    col for col in gemma.columns
    if col not in ["observation_date"] + macro_cols
]

print("DEEPSEEK GEOPOLITICAL FEATURES:")
print(deepseek_geop_cols)

print("\nGEMMA GEOPOLITICAL FEATURES:")
print(gemma_geop_cols)

print("\nDeepSeek geopolitical feature count:",
      len(deepseek_geop_cols))

print("Gemma geopolitical feature count:",
      len(gemma_geop_cols))

DEEPSEEK GEOPOLITICAL FEATURES:
['trade_sum', 'sanctions_sum', 'fed_pressure_sum', 'trade_max', 'sanctions_max', 'fed_pressure_max', 'trade_sum_ma3', 'sanctions_sum_ma3', 'fed_pressure_sum_ma3', 'trade_sum_ma5', 'sanctions_sum_ma5', 'fed_pressure_sum_ma5', 'trade_sum_ma10', 'sanctions_sum_ma10', 'fed_pressure_sum_ma10']

GEMMA GEOPOLITICAL FEATURES:
['trade_sum', 'sanctions_sum', 'fed_pressure_sum', 'trade_max', 'sanctions_max', 'fed_pressure_max', 'trade_sum_ma3', 'sanctions_sum_ma3', 'fed_pressure_sum_ma3', 'trade_sum_ma5', 'sanctions_sum_ma5', 'fed_pressure_sum_ma5', 'trade_sum_ma10', 'sanctions_sum_ma10', 'fed_pressure_sum_ma10']

DeepSeek geopolitical feature count: 15
Gemma geopolitical feature count: 15


In [7]:
# Extend geopolitical feature horizons to 30 and 60 trading days
#
# Rolling windows are backward-looking:
# each value at t uses information available up to and including t.

base_geop = [
    "trade_sum",
    "sanctions_sum",
    "fed_pressure_sum"
]

for df in [deepseek, gemma]:
    for col in base_geop:

        df[f"{col}_ma30"] = (
            df[col]
            .rolling(window=30, min_periods=30)
            .mean()
        )

        df[f"{col}_ma60"] = (
            df[col]
            .rolling(window=60, min_periods=60)
            .mean()
        )

new_long_features = (
    [f"{col}_ma30" for col in base_geop]
    + [f"{col}_ma60" for col in base_geop]
)

print("New longer-horizon features:")
print(new_long_features)

print("\nDeepSeek missing values:")
print(deepseek[new_long_features].isna().sum())

print("\nGemma missing values:")
print(gemma[new_long_features].isna().sum())

New longer-horizon features:
['trade_sum_ma30', 'sanctions_sum_ma30', 'fed_pressure_sum_ma30', 'trade_sum_ma60', 'sanctions_sum_ma60', 'fed_pressure_sum_ma60']

DeepSeek missing values:
trade_sum_ma30           29
sanctions_sum_ma30       29
fed_pressure_sum_ma30    29
trade_sum_ma60           59
sanctions_sum_ma60       59
fed_pressure_sum_ma60    59
dtype: int64

Gemma missing values:
trade_sum_ma30           29
sanctions_sum_ma30       29
fed_pressure_sum_ma30    29
trade_sum_ma60           59
sanctions_sum_ma60       59
fed_pressure_sum_ma60    59
dtype: int64


In [8]:
# Define and verify the complete geopolitical candidate feature set

geop_windows = [
    "trade_sum",
    "sanctions_sum",
    "fed_pressure_sum",
    
    "trade_max",
    "sanctions_max",
    "fed_pressure_max",
    
    "trade_sum_ma3",
    "sanctions_sum_ma3",
    "fed_pressure_sum_ma3",
    
    "trade_sum_ma5",
    "sanctions_sum_ma5",
    "fed_pressure_sum_ma5",
    
    "trade_sum_ma10",
    "sanctions_sum_ma10",
    "fed_pressure_sum_ma10",
    
    "trade_sum_ma30",
    "sanctions_sum_ma30",
    "fed_pressure_sum_ma30",
    
    "trade_sum_ma60",
    "sanctions_sum_ma60",
    "fed_pressure_sum_ma60"
]

print("Number of geopolitical candidate features:", len(geop_windows))

print("\nAll present in DeepSeek:",
      all(col in deepseek.columns for col in geop_windows))

print("All present in Gemma:",
      all(col in gemma.columns for col in geop_windows))

print("\nCandidate features:")
for col in geop_windows:
    print(col)

Number of geopolitical candidate features: 21

All present in DeepSeek: True
All present in Gemma: True

Candidate features:
trade_sum
sanctions_sum
fed_pressure_sum
trade_max
sanctions_max
fed_pressure_max
trade_sum_ma3
sanctions_sum_ma3
fed_pressure_sum_ma3
trade_sum_ma5
sanctions_sum_ma5
fed_pressure_sum_ma5
trade_sum_ma10
sanctions_sum_ma10
fed_pressure_sum_ma10
trade_sum_ma30
sanctions_sum_ma30
fed_pressure_sum_ma30
trade_sum_ma60
sanctions_sum_ma60
fed_pressure_sum_ma60


In [9]:
# Audit usable sample for every target-horizon combination
#
# A row is usable only if:
#   1. all macro predictors are observed,
#   2. all DeepSeek geopolitical predictors are observed,
#   3. all Gemma geopolitical predictors are observed,
#   4. the relevant forward target is observed.
#
# This guarantees identical dates for the macro-only,
# DeepSeek and Gemma comparisons within each experiment.

macro_predictors = [
    "DGS2_diff",
    "USEPUINDXD_diff",
    "VIXCLS"
]

audit_rows = []

for target_type in ["return", "volatility", "direction"]:
    for h in HORIZONS:

        target_col = f"{target_type}_{h}d"

        valid_mask = (
            deepseek[macro_predictors + geop_windows].notna().all(axis=1)
            & gemma[macro_predictors + geop_windows].notna().all(axis=1)
            & targets[target_col].notna()
        )

        valid_dates = deepseek.loc[
            valid_mask, "observation_date"
        ]

        audit_rows.append({
            "target": target_type,
            "horizon": h,
            "n_obs": valid_mask.sum(),
            "start_date": valid_dates.min(),
            "end_date": valid_dates.max()
        })

sample_audit = pd.DataFrame(audit_rows)

display(sample_audit)

,target,horizon,n_obs,start_date,end_date
0,return,1,933,2017-04-06,2021-01-07
1,return,5,929,2017-04-06,2020-12-31
2,return,10,924,2017-04-06,2020-12-22
3,return,21,913,2017-04-06,2020-12-07
4,volatility,1,933,2017-04-06,2021-01-07
5,volatility,5,929,2017-04-06,2020-12-31
6,volatility,10,924,2017-04-06,2020-12-22
7,volatility,21,913,2017-04-06,2020-12-07
8,direction,1,933,2017-04-06,2021-01-07
9,direction,5,929,2017-04-06,2020-12-31


In [11]:
# Build clean experiment datasets for all 12 target-horizon combinations

experiments = {}

for target_type in ["return", "volatility", "direction"]:
    for h in HORIZONS:

        target_col = f"{target_type}_{h}d"

        # Common valid rows for this target/horizon
        valid_mask = (
            deepseek[macro_predictors + geop_windows].notna().all(axis=1)
            & gemma[macro_predictors + geop_windows].notna().all(axis=1)
            & targets[target_col].notna()
        )

        # Dates
        dates_h = deepseek.loc[
            valid_mask, "observation_date"
        ].reset_index(drop=True)

        # Macro-only predictors
        X_macro = deepseek.loc[
            valid_mask, macro_predictors
        ].reset_index(drop=True)

        # Macro + DeepSeek geopolitical predictors
        X_ds = deepseek.loc[
            valid_mask,
            macro_predictors + geop_windows
        ].reset_index(drop=True)

        # Macro + Gemma geopolitical predictors
        X_gm = gemma.loc[
            valid_mask,
            macro_predictors + geop_windows
        ].reset_index(drop=True)

        # Target
        y = targets.loc[
            valid_mask, target_col
        ].reset_index(drop=True)

        experiments[(target_type, h)] = {
            "dates": dates_h,
            "X_macro": X_macro,
            "X_ds": X_ds,
            "X_gm": X_gm,
            "y": y
        }

print("Number of experiments:", len(experiments))

print("\nExperiment sizes:")
for (target_type, h), exp in experiments.items():
    print(
        f"{target_type:10s} {h:2d}d | "
        f"N={len(exp['y']):3d} | "
        f"macro={exp['X_macro'].shape[1]:2d} | "
        f"DeepSeek={exp['X_ds'].shape[1]:2d} | "
        f"Gemma={exp['X_gm'].shape[1]:2d}"
    )

Number of experiments: 12

Experiment sizes:
return      1d | N=933 | macro= 3 | DeepSeek=24 | Gemma=24
return      5d | N=929 | macro= 3 | DeepSeek=24 | Gemma=24
return     10d | N=924 | macro= 3 | DeepSeek=24 | Gemma=24
return     21d | N=913 | macro= 3 | DeepSeek=24 | Gemma=24
volatility  1d | N=933 | macro= 3 | DeepSeek=24 | Gemma=24
volatility  5d | N=929 | macro= 3 | DeepSeek=24 | Gemma=24
volatility 10d | N=924 | macro= 3 | DeepSeek=24 | Gemma=24
volatility 21d | N=913 | macro= 3 | DeepSeek=24 | Gemma=24
direction   1d | N=933 | macro= 3 | DeepSeek=24 | Gemma=24
direction   5d | N=929 | macro= 3 | DeepSeek=24 | Gemma=24
direction  10d | N=924 | macro= 3 | DeepSeek=24 | Gemma=24
direction  21d | N=913 | macro= 3 | DeepSeek=24 | Gemma=24


In [12]:
# Create horizon-aware chronological train/test splits
#
# For an h-day forward target, observations immediately before the
# test period have target windows extending into the test period.
# We therefore purge h observations between training and test.

TEST_SIZE = 0.20

for (target_type, h), exp in experiments.items():

    n = len(exp["y"])

    # First observation belonging to the final 20% test period
    test_start = int(np.floor(n * (1 - TEST_SIZE)))

    # Purge h observations before test
    train_end = test_start - h

    exp["train_idx"] = np.arange(0, train_end)
    exp["test_idx"] = np.arange(test_start, n)

    # Store purged observations for audit
    exp["purged_idx"] = np.arange(train_end, test_start)

print("Chronological split audit:\n")

for (target_type, h), exp in experiments.items():

    train_idx = exp["train_idx"]
    test_idx = exp["test_idx"]
    purged_idx = exp["purged_idx"]

    print(
        f"{target_type:10s} {h:2d}d | "
        f"train={len(train_idx):3d} | "
        f"purged={len(purged_idx):2d} | "
        f"test={len(test_idx):3d} | "
        f"train end={exp['dates'].iloc[train_idx[-1]].date()} | "
        f"test start={exp['dates'].iloc[test_idx[0]].date()}"
    )

Chronological split audit:

return      1d | train=745 | purged= 1 | test=187 | train end=2020-04-03 | test start=2020-04-07
return      5d | train=738 | purged= 5 | test=186 | train end=2020-03-25 | test start=2020-04-02
return     10d | train=729 | purged=10 | test=185 | train end=2020-03-12 | test start=2020-03-27
return     21d | train=709 | purged=21 | test=183 | train end=2020-02-12 | test start=2020-03-16
volatility  1d | train=745 | purged= 1 | test=187 | train end=2020-04-03 | test start=2020-04-07
volatility  5d | train=738 | purged= 5 | test=186 | train end=2020-03-25 | test start=2020-04-02
volatility 10d | train=729 | purged=10 | test=185 | train end=2020-03-12 | test start=2020-03-27
volatility 21d | train=709 | purged=21 | test=183 | train end=2020-02-12 | test start=2020-03-16
direction   1d | train=745 | purged= 1 | test=187 | train end=2020-04-03 | test start=2020-04-07
direction   5d | train=738 | purged= 5 | test=186 | train end=2020-03-25 | test start=2020-04-02
di

In [13]:
# Horizon-aware time-series cross-validation
#
# The CV gap equals the forward forecast horizon so that
# training target windows do not overlap validation periods.

N_SPLITS = 5

cv_by_horizon = {
    h: TimeSeriesSplit(
        n_splits=N_SPLITS,
        gap=h
    )
    for h in HORIZONS
}

# Audit the CV structure using the RETURN experiments.
# Sample sizes are identical across target families at each horizon.

for h in HORIZONS:

    exp = experiments[("return", h)]
    train_idx = exp["train_idx"]

    X_train_audit = exp["X_macro"].iloc[train_idx]

    print(f"\n--- {h}-DAY HORIZON ---")

    for fold, (cv_train, cv_val) in enumerate(
        cv_by_horizon[h].split(X_train_audit),
        start=1
    ):
        print(
            f"Fold {fold}: "
            f"train={len(cv_train)}, "
            f"gap={cv_val[0] - cv_train[-1] - 1}, "
            f"validation={len(cv_val)}"
        )


--- 1-DAY HORIZON ---
Fold 1: train=124, gap=1, validation=124
Fold 2: train=248, gap=1, validation=124
Fold 3: train=372, gap=1, validation=124
Fold 4: train=496, gap=1, validation=124
Fold 5: train=620, gap=1, validation=124

--- 5-DAY HORIZON ---
Fold 1: train=118, gap=5, validation=123
Fold 2: train=241, gap=5, validation=123
Fold 3: train=364, gap=5, validation=123
Fold 4: train=487, gap=5, validation=123
Fold 5: train=610, gap=5, validation=123

--- 10-DAY HORIZON ---
Fold 1: train=114, gap=10, validation=121
Fold 2: train=235, gap=10, validation=121
Fold 3: train=356, gap=10, validation=121
Fold 4: train=477, gap=10, validation=121
Fold 5: train=598, gap=10, validation=121

--- 21-DAY HORIZON ---
Fold 1: train=98, gap=21, validation=118
Fold 2: train=216, gap=21, validation=118
Fold 3: train=334, gap=21, validation=118
Fold 4: train=452, gap=21, validation=118
Fold 5: train=570, gap=21, validation=118


In [14]:
# Regularised regression setup for return and volatility targets
#
# Scaling occurs INSIDE the pipeline, so each CV training fold
# fits its own scaler and there is no scaling leakage.

lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=100000))
])

# Same broad penalty range used in the audited LASSO analysis
lasso_alphas = np.logspace(-6, 0, 200)

lasso_param_grid = {
    "lasso__alpha": lasso_alphas
}

print("Regression estimator: LASSO")
print("Number of alpha values:", len(lasso_alphas))
print(
    "Alpha range:",
    lasso_alphas.min(),
    "to",
    lasso_alphas.max()
)

Regression estimator: LASSO
Number of alpha values: 200
Alpha range: 1e-06 to 1.0


In [23]:
# L1-regularised logistic regression setup for direction targets
#
# In current scikit-learn versions, l1_ratio=1.0 gives a pure
# L1 penalty (classification analogue of LASSO).

logit_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(
        solver="liblinear",
        l1_ratio=1.0,
        max_iter=100000
    ))
])

# C is inverse regularisation strength:
# smaller C = stronger regularisation
# larger C = weaker regularisation

logit_Cs = np.logspace(-4, 4, 100)

logit_param_grid = {
    "logit__C": logit_Cs
}

print("Classification estimator: L1 Logistic Regression")
print("Number of C values:", len(logit_Cs))
print(
    "C range:",
    logit_Cs.min(),
    "to",
    logit_Cs.max()
)

Classification estimator: L1 Logistic Regression
Number of C values: 100
C range: 0.0001 to 10000.0


In [24]:
# Evaluation functions for the target matrix

def regression_metrics(y_true, y_pred):
    """Metrics for return and realised-volatility targets."""
    
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }


def classification_metrics(y_true, y_pred, y_prob):
    """Metrics for direction targets."""
    
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_prob)
    }

In [ ]:
# Run one return/volatility target-horizon experiment

def run_regression_experiment(target_type, h):

    exp = experiments[(target_type, h)]

    train_idx = exp["train_idx"]
    test_idx = exp["test_idx"]

    y_train = exp["y"].iloc[train_idx]
    y_test = exp["y"].iloc[test_idx]

    # Predictor sets
    X_macro_train = exp["X_macro"].iloc[train_idx]
    X_macro_test = exp["X_macro"].iloc[test_idx]

    X_ds_train = exp["X_ds"].iloc[train_idx]
    X_ds_test = exp["X_ds"].iloc[test_idx]

    X_gm_train = exp["X_gm"].iloc[train_idx]
    X_gm_test = exp["X_gm"].iloc[test_idx]

    # Horizon-aware CV
    cv = cv_by_horizon[h]

    
    # Naive benchmark
    benchmark_pred = np.repeat(
        y_train.mean(),
        len(y_test)
    )

    
    # Macro-only LASSO
    macro_model = GridSearchCV(
        estimator=lasso_pipe,
        param_grid=lasso_param_grid,
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )

    macro_model.fit(X_macro_train, y_train)
    macro_pred = macro_model.predict(X_macro_test)

    
    # DeepSeek LASSO
    ds_model = GridSearchCV(
        estimator=lasso_pipe,
        param_grid=lasso_param_grid,
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )

    ds_model.fit(X_ds_train, y_train)
    ds_pred = ds_model.predict(X_ds_test)

    
    # Gemma LASSO
    gm_model = GridSearchCV(
        estimator=lasso_pipe,
        param_grid=lasso_param_grid,
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )

    gm_model.fit(X_gm_train, y_train)
    gm_pred = gm_model.predict(X_gm_test)

    
    # Evaluation
    results = {
        "Benchmark": regression_metrics(y_test, benchmark_pred),
        "Macro": regression_metrics(y_test, macro_pred),
        "DeepSeek": regression_metrics(y_test, ds_pred),
        "Gemma": regression_metrics(y_test, gm_pred)
    }

    # Store useful diagnostics
    models = {
        "Macro": macro_model,
        "DeepSeek": ds_model,
        "Gemma": gm_model
    }

    predictions = {
        "y_test": y_test.reset_index(drop=True),
        "Benchmark": benchmark_pred,
        "Macro": macro_pred,
        "DeepSeek": ds_pred,
        "Gemma": gm_pred
    }

    return results, models, predictions

In [ ]:
# Run one direction target-horizon experiment

def run_classification_experiment(h):

    exp = experiments[("direction", h)]

    train_idx = exp["train_idx"]
    test_idx = exp["test_idx"]

    y_train = exp["y"].iloc[train_idx].astype(int)
    y_test = exp["y"].iloc[test_idx].astype(int)

    # Predictor sets
    X_macro_train = exp["X_macro"].iloc[train_idx]
    X_macro_test = exp["X_macro"].iloc[test_idx]

    X_ds_train = exp["X_ds"].iloc[train_idx]
    X_ds_test = exp["X_ds"].iloc[test_idx]

    X_gm_train = exp["X_gm"].iloc[train_idx]
    X_gm_test = exp["X_gm"].iloc[test_idx]

    # Horizon-aware CV
    cv = cv_by_horizon[h]

    
    # Naive benchmark
    majority_class = int(y_train.mean() >= 0.5)
    benchmark_prob = np.repeat(
        y_train.mean(),
        len(y_test)
    )
    benchmark_pred = np.repeat(
        majority_class,
        len(y_test)
    )

    
    # Macro-only L1 logistic
    macro_model = GridSearchCV(
        estimator=logit_pipe,
        param_grid=logit_param_grid,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    macro_model.fit(X_macro_train, y_train)

    macro_pred = macro_model.predict(X_macro_test)
    macro_prob = macro_model.predict_proba(X_macro_test)[:, 1]

    
    # DeepSeek L1 logistic
    ds_model = GridSearchCV(
        estimator=logit_pipe,
        param_grid=logit_param_grid,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    ds_model.fit(X_ds_train, y_train)

    ds_pred = ds_model.predict(X_ds_test)
    ds_prob = ds_model.predict_proba(X_ds_test)[:, 1]

    # Gemma L1 logistic
    gm_model = GridSearchCV(
        estimator=logit_pipe,
        param_grid=logit_param_grid,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    gm_model.fit(X_gm_train, y_train)

    gm_pred = gm_model.predict(X_gm_test)
    gm_prob = gm_model.predict_proba(X_gm_test)[:, 1]

    # Evaluation
    results = {
        "Benchmark": classification_metrics(
            y_test, benchmark_pred, benchmark_prob
        ),
        "Macro": classification_metrics(
            y_test, macro_pred, macro_prob
        ),
        "DeepSeek": classification_metrics(
            y_test, ds_pred, ds_prob
        ),
        "Gemma": classification_metrics(
            y_test, gm_pred, gm_prob
        )
    }

    models = {
        "Macro": macro_model,
        "DeepSeek": ds_model,
        "Gemma": gm_model
    }

    predictions = {
        "y_test": y_test.reset_index(drop=True),
        "Benchmark_pred": benchmark_pred,
        "Benchmark_prob": benchmark_prob,
        "Macro_pred": macro_pred,
        "Macro_prob": macro_prob,
        "DeepSeek_pred": ds_pred,
        "DeepSeek_prob": ds_prob,
        "Gemma_pred": gm_pred,
        "Gemma_prob": gm_prob
    }

    return results, models, predictions

In [27]:
# Sanity check: 1-day realised-volatility experiment

vol1_results, vol1_models, vol1_predictions = (
    run_regression_experiment(
        target_type="volatility",
        h=1
    )
)

vol1_table = pd.DataFrame(vol1_results).T

display(vol1_table)

print("\nSelected alphas:")
for model_name, model in vol1_models.items():
    print(
        f"{model_name}:",
        model.best_params_["lasso__alpha"]
    )

print("\nSelected non-zero predictors:")
for model_name, model in vol1_models.items():

    feature_names = (
        experiments[("volatility", 1)]["X_macro"].columns
        if model_name == "Macro"
        else experiments[("volatility", 1)][
            "X_ds" if model_name == "DeepSeek" else "X_gm"
        ].columns
    )

    coefs = model.best_estimator_[
        "lasso"
    ].coef_

    selected = [
        feature
        for feature, coef in zip(feature_names, coefs)
        if not np.isclose(coef, 0)
    ]

    print(f"{model_name}: {selected}")

,RMSE,MAE,R2
Benchmark,0.002457,0.001954,-0.003181
Macro,0.002488,0.002082,-0.028049
DeepSeek,0.002463,0.002041,-0.008009
Gemma,0.002482,0.002080,-0.023419



Selected alphas:
Macro: 0.00017027691722258995
DeepSeek: 0.0003409285069746811
Gemma: 0.0002768286630392064

Selected non-zero predictors:
Macro: ['VIXCLS']
DeepSeek: ['VIXCLS', 'trade_sum_ma60']
Gemma: ['VIXCLS', 'trade_sum_ma30', 'sanctions_sum_ma30', 'trade_sum_ma60', 'fed_pressure_sum_ma60']


In [ ]:
# Run the complete target-horizon matrix

all_results = {}
all_models = {}
all_predictions = {}

# Return and volatility regression

for target_type in ["return", "volatility"]:
    for h in HORIZONS:

        print(f"Running {target_type} - {h}d...")

        results, models, predictions = (
            run_regression_experiment(
                target_type=target_type,
                h=h
            )
        )

        all_results[(target_type, h)] = results
        all_models[(target_type, h)] = models
        all_predictions[(target_type, h)] = predictions


# Direction classification

for h in HORIZONS:

    print(f"Running direction - {h}d...")

    results, models, predictions = (
        run_classification_experiment(h=h)
    )

    all_results[("direction", h)] = results
    all_models[("direction", h)] = models
    all_predictions[("direction", h)] = predictions


print("\nComplete.")
print("Number of target-horizon experiments:", len(all_results))

Running return - 1d...
Running return - 5d...
Running return - 10d...
Running return - 21d...
Running volatility - 1d...
Running volatility - 5d...
Running volatility - 10d...
Running volatility - 21d...
Running direction - 1d...
Running direction - 5d...
Running direction - 10d...
Running direction - 21d...

Complete.
Number of target-horizon experiments: 12


In [29]:
# Build the final target-horizon performance matrix
#
# Regression targets:
#   primary metric = RMSE (lower is better)
#
# Direction target:
#   primary metric = ROC-AUC (higher is better)
#
# We compare:
#   naive benchmark
#   macro-only
#   macro + DeepSeek
#   macro + Gemma

matrix_rows = []

for target_type in ["return", "volatility", "direction"]:
    for h in HORIZONS:

        res = all_results[(target_type, h)]

        if target_type in ["return", "volatility"]:

            benchmark = res["Benchmark"]["RMSE"]
            macro = res["Macro"]["RMSE"]
            deepseek_score = res["DeepSeek"]["RMSE"]
            gemma_score = res["Gemma"]["RMSE"]

            # Positive improvement = lower RMSE = better
            macro_vs_benchmark = 100 * (
                benchmark - macro
            ) / benchmark

            ds_vs_macro = 100 * (
                macro - deepseek_score
            ) / macro

            gm_vs_macro = 100 * (
                macro - gemma_score
            ) / macro

            ds_beats_macro = deepseek_score < macro
            gm_beats_macro = gemma_score < macro

            metric = "RMSE"

        else:

            benchmark = res["Benchmark"]["ROC_AUC"]
            macro = res["Macro"]["ROC_AUC"]
            deepseek_score = res["DeepSeek"]["ROC_AUC"]
            gemma_score = res["Gemma"]["ROC_AUC"]

            # For AUC, report changes in percentage points
            macro_vs_benchmark = 100 * (
                macro - benchmark
            )

            ds_vs_macro = 100 * (
                deepseek_score - macro
            )

            gm_vs_macro = 100 * (
                gemma_score - macro
            )

            ds_beats_macro = deepseek_score > macro
            gm_beats_macro = gemma_score > macro

            metric = "ROC-AUC"

        matrix_rows.append({
            "Target": target_type,
            "Horizon": h,
            "Metric": metric,
            "Benchmark": benchmark,
            "Macro": macro,
            "DeepSeek": deepseek_score,
            "Gemma": gemma_score,
            "Macro_vs_Benchmark": macro_vs_benchmark,
            "DeepSeek_vs_Macro": ds_vs_macro,
            "Gemma_vs_Macro": gm_vs_macro,
            "DeepSeek_beats_Macro": ds_beats_macro,
            "Gemma_beats_Macro": gm_beats_macro
        })

target_matrix = pd.DataFrame(matrix_rows)

display(target_matrix)

,Target,Horizon,Metric,Benchmark,Macro,DeepSeek,Gemma,Macro_vs_Benchmark,DeepSeek_vs_Macro,Gemma_vs_Macro,DeepSeek_beats_Macro,Gemma_beats_Macro
0,return,1,RMSE,0.004060,0.004060,0.004060,0.004060,0.000000,0.000000,0.000000,False,False
1,return,5,RMSE,0.009543,0.009543,0.009543,0.009379,0.000000,0.000000,1.726642,False,True
2,return,10,RMSE,0.013514,0.013652,0.013514,0.013843,-1.019292,1.009007,-1.400175,True,False
3,return,21,RMSE,0.022001,0.022001,0.022001,0.021772,0.000000,0.000000,1.041423,False,True
4,volatility,1,RMSE,0.002457,0.002488,0.002463,0.002482,-1.231859,0.979442,0.225419,True,True
5,volatility,5,RMSE,0.002740,0.003045,0.003163,0.003045,-11.118711,-3.881062,0.012735,False,True
6,volatility,10,RMSE,0.002677,0.002470,0.002901,0.002591,7.724550,-17.453947,-4.922008,False,False
7,volatility,21,RMSE,0.003916,0.004015,0.003875,0.003841,-2.530841,3.492049,4.355254,True,True
8,direction,1,ROC-AUC,0.500000,0.471336,0.517453,0.466828,-2.866389,4.611650,-0.450763,True,False
9,direction,5,ROC-AUC,0.500000,0.500000,0.543161,0.516778,0.000000,4.316086,1.677766,True,True


In [30]:
# Inspect selected predictors and regularisation parameters
# for the most informative target-horizon combinations

interesting_cases = [
    ("return", 5),
    ("return", 21),
    ("volatility", 10),
    ("volatility", 21),
    ("direction", 1),
    ("direction", 5),
    ("direction", 10),
]


def get_feature_names(target_type, h, model_name):

    exp = experiments[(target_type, h)]

    if model_name == "Macro":
        return exp["X_macro"].columns

    elif model_name == "DeepSeek":
        return exp["X_ds"].columns

    elif model_name == "Gemma":
        return exp["X_gm"].columns


for target_type, h in interesting_cases:

    print("\n" + "=" * 75)
    print(f"{target_type.upper()} — {h}-DAY HORIZON")
    print("=" * 75)

    models = all_models[(target_type, h)]

    for model_name in ["Macro", "DeepSeek", "Gemma"]:

        model = models[model_name]
        feature_names = get_feature_names(
            target_type, h, model_name
        )

        if target_type in ["return", "volatility"]:

            estimator = model.best_estimator_.named_steps["lasso"]
            coefs = estimator.coef_

            tuning_name = "alpha"
            tuning_value = model.best_params_["lasso__alpha"]

        else:

            estimator = model.best_estimator_.named_steps["logit"]
            coefs = estimator.coef_.ravel()

            tuning_name = "C"
            tuning_value = model.best_params_["logit__C"]

        coef_table = pd.DataFrame({
            "predictor": feature_names,
            "coefficient": coefs
        })

        # L1 models: retain genuinely non-zero coefficients
        selected = (
            coef_table[
                ~np.isclose(
                    coef_table["coefficient"],
                    0.0,
                    atol=1e-12
                )
            ]
            .assign(
                abs_coefficient=lambda x:
                    x["coefficient"].abs()
            )
            .sort_values(
                "abs_coefficient",
                ascending=False
            )
            .drop(columns="abs_coefficient")
        )

        print(f"\n{model_name}")
        print(f"Best {tuning_name}: {tuning_value}")
        print(f"Number selected: {len(selected)}")

        if len(selected) == 0:
            print("No predictors selected.")
        else:
            display(selected.reset_index(drop=True))


RETURN — 5-DAY HORIZON

Macro
Best alpha: 0.0010353218432956617
Number selected: 0
No predictors selected.

DeepSeek
Best alpha: 0.002221946860939524
Number selected: 0
No predictors selected.

Gemma
Best alpha: 0.0012750512407130128
Number selected: 3


,predictor,coefficient
0,sanctions_sum_ma5,-0.000397
1,sanctions_sum_ma60,-0.000134
2,trade_sum_ma60,-0.000008



RETURN — 21-DAY HORIZON

Macro
Best alpha: 0.004768611697714469
Number selected: 0
No predictors selected.

DeepSeek
Best alpha: 0.00890735463861044
Number selected: 0
No predictors selected.

Gemma
Best alpha: 0.004448782831127585
Number selected: 2


,predictor,coefficient
0,trade_sum_ma60,-0.000937
1,trade_sum_ma30,-0.000132



VOLATILITY — 10-DAY HORIZON

Macro
Best alpha: 0.0007842822061337682
Number selected: 1


...

,predictor,coefficient
0,VIXCLS,-0.081838



DeepSeek
Best C: 0.015199110829529346
Number selected: 2


,predictor,coefficient
0,sanctions_sum_ma30,-0.059654
1,sanctions_sum_ma60,-0.045127



Gemma
Best C: 0.026560877829466867
Number selected: 3


,predictor,coefficient
0,sanctions_sum_ma60,-0.136128
1,trade_sum_ma60,-0.099286
2,VIXCLS,-0.026246


In [31]:
# Statistical comparison of 21-day realised-volatility forecasts
# HAC standard errors account for overlap in the 21-day forward targets.

import statsmodels.api as sm

target_type = "volatility"
h = 21

preds = all_predictions[(target_type, h)]

y_test = np.asarray(preds["y_test"])

forecast_dict = {
    "Benchmark": np.asarray(preds["Benchmark"]),
    "Macro": np.asarray(preds["Macro"]),
    "DeepSeek": np.asarray(preds["DeepSeek"]),
    "Gemma": np.asarray(preds["Gemma"])
}


def dm_hac_test(y, pred_a, pred_b, maxlags):
    """
    Compare forecast A against forecast B using squared-error loss.

    Positive mean_loss_diff:
        A has larger MSE than B -> B is better.

    HAC/Newey-West SE accounts for serial correlation.
    """

    loss_a = (y - pred_a) ** 2
    loss_b = (y - pred_b) ** 2

    d = loss_a - loss_b

    X = np.ones((len(d), 1))

    model = sm.OLS(d, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags}
    )

    return {
        "mean_loss_diff": d.mean(),
        "t_stat": model.tvalues[0],
        "p_value": model.pvalues[0]
    }


comparisons = [
    ("Macro", "DeepSeek"),
    ("Macro", "Gemma"),
    ("Benchmark", "DeepSeek"),
    ("Benchmark", "Gemma"),
    ("DeepSeek", "Gemma")
]

dm_rows = []

for a, b in comparisons:

    result = dm_hac_test(
        y_test,
        forecast_dict[a],
        forecast_dict[b],
        maxlags=h - 1
    )

    dm_rows.append({
        "Model_A": a,
        "Model_B": b,
        "Mean_loss_A_minus_B": result["mean_loss_diff"],
        "t_stat": result["t_stat"],
        "p_value": result["p_value"]
    })

dm_results = pd.DataFrame(dm_rows)

display(dm_results)

,Model_A,Model_B,Mean_loss_A_minus_B,t_stat,p_value
0,Macro,DeepSeek,1.106452e-06,1.924205,0.054329
1,Macro,Gemma,1.373896e-06,2.904107,0.003683
2,Benchmark,DeepSeek,3.202771e-07,0.431711,0.665951
3,Benchmark,Gemma,5.877210e-07,1.412962,0.157667
4,DeepSeek,Gemma,2.674439e-07,0.694539,0.487344


In [32]:
# Formal forecast comparison for the promising Gemma return results
# HAC lag = h - 1 to account for overlapping h-day forward returns

return_test_rows = []

for h in [5, 21]:

    preds = all_predictions[("return", h)]

    y_test = np.asarray(preds["y_test"])

    forecast_dict = {
        "Benchmark": np.asarray(preds["Benchmark"]),
        "Macro": np.asarray(preds["Macro"]),
        "DeepSeek": np.asarray(preds["DeepSeek"]),
        "Gemma": np.asarray(preds["Gemma"])
    }

    comparisons = [
        ("Benchmark", "Gemma"),
        ("Macro", "Gemma"),
        ("DeepSeek", "Gemma")
    ]

    for model_a, model_b in comparisons:

        result = dm_hac_test(
            y_test,
            forecast_dict[model_a],
            forecast_dict[model_b],
            maxlags=h - 1
        )

        return_test_rows.append({
            "Horizon": h,
            "Model_A": model_a,
            "Model_B": model_b,
            "Mean_loss_A_minus_B": result["mean_loss_diff"],
            "t_stat": result["t_stat"],
            "p_value": result["p_value"]
        })

return_dm_results = pd.DataFrame(return_test_rows)

display(return_dm_results)

,Horizon,Model_A,Model_B,Mean_loss_A_minus_B,t_stat,p_value
0,5,Benchmark,Gemma,0.000003,2.626881,0.008617
1,5,Macro,Gemma,0.000003,2.626881,0.008617
2,5,DeepSeek,Gemma,0.000003,2.626881,0.008617
3,21,Benchmark,Gemma,0.000010,1.475659,0.140035
4,21,Macro,Gemma,0.000010,1.475659,0.140035
5,21,DeepSeek,Gemma,0.000010,1.475659,0.140035


In [33]:
# Moving-block bootstrap test for 5-day direction AUC differences
# Preserves short-range dependence from overlapping forward targets.

from sklearn.metrics import roc_auc_score

target_type = "direction"
h = 5

preds = all_predictions[(target_type, h)]

y_test = np.asarray(preds["y_test"])

probabilities = {
    "Benchmark": np.asarray(preds["Benchmark_prob"]),
    "Macro": np.asarray(preds["Macro_prob"]),
    "DeepSeek": np.asarray(preds["DeepSeek_prob"]),
    "Gemma": np.asarray(preds["Gemma_prob"])
}


def moving_block_auc_test(
    y,
    prob_a,
    prob_b,
    block_length=5,
    n_boot=5000,
    seed=42
):
    """
    Bootstrap the AUC difference:
        AUC(B) - AUC(A)

    Positive difference means model B performs better.
    """

    rng = np.random.default_rng(seed)
    n = len(y)

    observed_diff = (
        roc_auc_score(y, prob_b)
        - roc_auc_score(y, prob_a)
    )

    boot_diffs = []

    possible_starts = np.arange(
        0,
        n - block_length + 1
    )

    while len(boot_diffs) < n_boot:

        indices = []

        while len(indices) < n:
            start = rng.choice(possible_starts)

            block = np.arange(
                start,
                start + block_length
            )

            indices.extend(block.tolist())

        indices = np.asarray(indices[:n])

        y_b = y[indices]

        # AUC requires both classes
        if np.unique(y_b).size < 2:
            continue

        auc_a = roc_auc_score(
            y_b,
            prob_a[indices]
        )

        auc_b = roc_auc_score(
            y_b,
            prob_b[indices]
        )

        boot_diffs.append(
            auc_b - auc_a
        )

    boot_diffs = np.asarray(boot_diffs)

    ci_low, ci_high = np.percentile(
        boot_diffs,
        [2.5, 97.5]
    )

    # Two-sided bootstrap p-value around zero
    p_value = 2 * min(
        np.mean(boot_diffs <= 0),
        np.mean(boot_diffs >= 0)
    )

    return {
        "AUC_difference": observed_diff,
        "CI_95_low": ci_low,
        "CI_95_high": ci_high,
        "p_value": min(p_value, 1.0)
    }


direction_rows = []

for model_a, model_b in [
    ("Benchmark", "DeepSeek"),
    ("Macro", "DeepSeek"),
    ("Gemma", "DeepSeek")
]:

    result = moving_block_auc_test(
        y_test,
        probabilities[model_a],
        probabilities[model_b],
        block_length=h,
        n_boot=5000
    )

    direction_rows.append({
        "Model_A": model_a,
        "Model_B": model_b,
        **result
    })

direction_5d_tests = pd.DataFrame(direction_rows)

display(direction_5d_tests)

,Model_A,Model_B,AUC_difference,CI_95_low,CI_95_high,p_value
0,Benchmark,DeepSeek,0.043161,-0.079618,0.173114,0.4312
1,Macro,DeepSeek,0.043161,-0.079618,0.173114,0.4312
2,Gemma,DeepSeek,0.026383,-0.109182,0.161024,0.6896


In [34]:
# Build compact validation summary with evidence labels

summary_rows = []

for _, row in target_matrix.iterrows():

    target = row["Target"]
    h = int(row["Horizon"])

    ds_gain = row["DeepSeek_vs_Macro"]
    gm_gain = row["Gemma_vs_Macro"]

    ds_label = "Descriptive only"
    gm_label = "Descriptive only"

    # --------------------------------
    # Formally tested cases
    # --------------------------------

    if target == "return" and h == 5:
        ds_label = "No gain"
        gm_label = "Significant improvement"

    elif target == "return" and h == 21:
        ds_label = "No gain"
        gm_label = "Improvement not significant"

    elif target == "volatility" and h == 21:
        ds_label = "Borderline vs macro"
        gm_label = "Significant vs macro only"

    elif target == "direction" and h == 5:
        ds_label = "Improvement not significant"
        gm_label = "Improvement not significant"

    else:
        if not row["DeepSeek_beats_Macro"]:
            ds_label = "No improvement"

        if not row["Gemma_beats_Macro"]:
            gm_label = "No improvement"

    summary_rows.append({
        "Target": target,
        "Horizon": h,
        "Metric": row["Metric"],
        "Macro": row["Macro"],
        "DeepSeek": row["DeepSeek"],
        "Gemma": row["Gemma"],
        "DeepSeek_vs_Macro": ds_gain,
        "Gemma_vs_Macro": gm_gain,
        "DeepSeek_evidence": ds_label,
        "Gemma_evidence": gm_label
    })

validation_summary = pd.DataFrame(summary_rows)

display(validation_summary)

,Target,Horizon,Metric,Macro,DeepSeek,Gemma,DeepSeek_vs_Macro,Gemma_vs_Macro,DeepSeek_evidence,Gemma_evidence
0,return,1,RMSE,0.004060,0.004060,0.004060,0.000000,0.000000,No improvement,No improvement
1,return,5,RMSE,0.009543,0.009543,0.009379,0.000000,1.726642,No gain,Significant improvement
2,return,10,RMSE,0.013652,0.013514,0.013843,1.009007,-1.400175,Descriptive only,No improvement
3,return,21,RMSE,0.022001,0.022001,0.021772,0.000000,1.041423,No gain,Improvement not significant
4,volatility,1,RMSE,0.002488,0.002463,0.002482,0.979442,0.225419,Descriptive only,Descriptive only
5,volatility,5,RMSE,0.003045,0.003163,0.003045,-3.881062,0.012735,No improvement,Descriptive only
6,volatility,10,RMSE,0.002470,0.002901,0.002591,-17.453947,-4.922008,No improvement,No improvement
7,volatility,21,RMSE,0.004015,0.003875,0.003841,3.492049,4.355254,Borderline vs macro,Significant vs macro only
8,direction,1,ROC-AUC,0.471336,0.517453,0.466828,4.611650,-0.450763,Descriptive only,No improvement
9,direction,5,ROC-AUC,0.500000,0.543161,0.516778,4.316086,1.677766,Improvement not significant,Improvement not significant


In [35]:
# FINAL MATRIX SPECIFICATION
# Use the exact established candidate-feature architecture
# from the regularisation analysis.

CANDIDATE_DIR = Path("../outputs/candidate_features")

candidate_ds = pd.read_csv(
    CANDIDATE_DIR / "candidate_features_deepseek.csv",
    parse_dates=["observation_date"]
)

candidate_gm = pd.read_csv(
    CANDIDATE_DIR / "candidate_features_gemma.csv",
    parse_dates=["observation_date"]
)

manifest = pd.read_csv(
    CANDIDATE_DIR / "candidate_feature_manifest.csv"
)

# Exact established macro/EURUSD benchmark predictors
macro_predictors_final = manifest.loc[
    manifest["source"].isin(["Macro", "EURUSD"]),
    "feature"
].tolist()

# Exact full candidate set used in the feature-selection analysis
established_predictors = manifest["feature"].tolist()

print("Candidate DeepSeek shape:", candidate_ds.shape)
print("Candidate Gemma shape:", candidate_gm.shape)

print("\nEstablished predictor count:",
      len(established_predictors))

print("Macro/EURUSD predictor count:",
      len(macro_predictors_final))

print("\nDeepSeek dates:",
      candidate_ds["observation_date"].min(),
      "to",
      candidate_ds["observation_date"].max())

print("Gemma dates:",
      candidate_gm["observation_date"].min(),
      "to",
      candidate_gm["observation_date"].max())

print(
    "\nDates identical:",
    candidate_ds["observation_date"].equals(
        candidate_gm["observation_date"]
    )
)

Candidate DeepSeek shape: (982, 126)
Candidate Gemma shape: (982, 126)

Established predictor count: 124
Macro/EURUSD predictor count: 43

DeepSeek dates: 2017-01-17 00:00:00 to 2021-01-08 00:00:00
Gemma dates: 2017-01-17 00:00:00 to 2021-01-08 00:00:00

Dates identical: True


In [36]:
# Add only the new MA30/MA60 geopolitical features
# to the established candidate-feature datasets.

long_geop_features = [
    "trade_sum_ma30",
    "sanctions_sum_ma30",
    "fed_pressure_sum_ma30",
    "trade_sum_ma60",
    "sanctions_sum_ma60",
    "fed_pressure_sum_ma60"
]

# Merge longer-horizon geopolitical features by date
final_ds = candidate_ds.merge(
    deepseek[["observation_date"] + long_geop_features],
    on="observation_date",
    how="left",
    validate="one_to_one"
)

final_gm = candidate_gm.merge(
    gemma[["observation_date"] + long_geop_features],
    on="observation_date",
    how="left",
    validate="one_to_one"
)

# Merge the already validated 12 forward targets
final_ds = final_ds.merge(
    targets,
    on="observation_date",
    how="left",
    validate="one_to_one"
)

final_gm = final_gm.merge(
    targets,
    on="observation_date",
    how="left",
    validate="one_to_one"
)

# Final predictor sets
full_predictors_final = (
    established_predictors
    + long_geop_features
)

print("Final DeepSeek shape:", final_ds.shape)
print("Final Gemma shape:", final_gm.shape)

print("\nMacro predictor count:",
      len(macro_predictors_final))

print("Full predictor count:",
      len(full_predictors_final))

print("\nAll macro predictors present in DeepSeek:",
      all(c in final_ds.columns for c in macro_predictors_final))

print("All full predictors present in DeepSeek:",
      all(c in final_ds.columns for c in full_predictors_final))

print("All full predictors present in Gemma:",
      all(c in final_gm.columns for c in full_predictors_final))

print("\nDates identical:",
      final_ds["observation_date"].equals(
          final_gm["observation_date"]
      ))

Final DeepSeek shape: (982, 144)
Final Gemma shape: (982, 144)

Macro predictor count: 43
Full predictor count: 130

All macro predictors present in DeepSeek: True
All full predictors present in DeepSeek: True
All full predictors present in Gemma: True

Dates identical: True


In [ ]:
# Build FINAL target-horizon experiments
# 43 macro/EURUSD predictors
# 130 full DeepSeek/Gemma predictors

experiments_final = {}

for target_type in ["return", "volatility", "direction"]:
    for h in HORIZONS:

        target_col = f"{target_type}_{h}d"

        # Common sample:
        # require every final predictor for both LLM datasets
        # plus the relevant forward target
        valid_mask = (
            final_ds[full_predictors_final].notna().all(axis=1)
            & final_gm[full_predictors_final].notna().all(axis=1)
            & final_ds[target_col].notna()
            & final_gm[target_col].notna()
        )

        dates_h = (
            final_ds.loc[valid_mask, "observation_date"]
            .reset_index(drop=True)
        )

        X_macro = (
            final_ds.loc[
                valid_mask,
                macro_predictors_final
            ]
            .reset_index(drop=True)
        )

        X_ds = (
            final_ds.loc[
                valid_mask,
                full_predictors_final
            ]
            .reset_index(drop=True)
        )

        X_gm = (
            final_gm.loc[
                valid_mask,
                full_predictors_final
            ]
            .reset_index(drop=True)
        )

        y = (
            final_ds.loc[
                valid_mask,
                target_col
            ]
            .reset_index(drop=True)
        )

        experiments_final[(target_type, h)] = {
            "dates": dates_h,
            "X_macro": X_macro,
            "X_ds": X_ds,
            "X_gm": X_gm,
            "y": y
        }


# Audit

audit_rows_final = []

for (target_type, h), exp in experiments_final.items():

    audit_rows_final.append({
        "target": target_type,
        "horizon": h,
        "n_obs": len(exp["y"]),
        "macro_p": exp["X_macro"].shape[1],
        "deepseek_p": exp["X_ds"].shape[1],
        "gemma_p": exp["X_gm"].shape[1],
        "start_date": exp["dates"].min(),
        "end_date": exp["dates"].max()
    })

final_sample_audit = pd.DataFrame(audit_rows_final)

display(final_sample_audit)

,target,horizon,n_obs,macro_p,deepseek_p,gemma_p,start_date,end_date
0,return,1,926,43,130,130,2017-04-06,2021-01-06
1,return,5,923,43,130,130,2017-04-06,2020-12-31
2,return,10,918,43,130,130,2017-04-06,2020-12-22
3,return,21,907,43,130,130,2017-04-06,2020-12-07
4,volatility,1,926,43,130,130,2017-04-06,2021-01-06
5,volatility,5,923,43,130,130,2017-04-06,2020-12-31
6,volatility,10,918,43,130,130,2017-04-06,2020-12-22
7,volatility,21,907,43,130,130,2017-04-06,2020-12-07
8,direction,1,926,43,130,130,2017-04-06,2021-01-06
9,direction,5,923,43,130,130,2017-04-06,2020-12-31


In [38]:
# FINAL horizon-aware chronological train/test splits

TEST_SIZE = 0.20

for (target_type, h), exp in experiments_final.items():

    n = len(exp["y"])

    test_start = int(np.floor(n * (1 - TEST_SIZE)))

    # Conservative purge equal to forecast horizon
    train_end = test_start - h

    exp["train_idx"] = np.arange(0, train_end)
    exp["purged_idx"] = np.arange(train_end, test_start)
    exp["test_idx"] = np.arange(test_start, n)


# Audit final splits
split_rows_final = []

for (target_type, h), exp in experiments_final.items():

    train_idx = exp["train_idx"]
    test_idx = exp["test_idx"]
    purged_idx = exp["purged_idx"]

    split_rows_final.append({
        "target": target_type,
        "horizon": h,
        "train_n": len(train_idx),
        "purged_n": len(purged_idx),
        "test_n": len(test_idx),
        "train_end": exp["dates"].iloc[train_idx[-1]],
        "test_start": exp["dates"].iloc[test_idx[0]]
    })

final_split_audit = pd.DataFrame(split_rows_final)

display(final_split_audit)

,target,horizon,train_n,purged_n,test_n,train_end,test_start
0,return,1,739,1,186,2020-04-03,2020-04-07
1,return,5,733,5,185,2020-03-26,2020-04-03
2,return,10,724,10,184,2020-03-13,2020-03-30
3,return,21,704,21,182,2020-02-13,2020-03-17
4,volatility,1,739,1,186,2020-04-03,2020-04-07
5,volatility,5,733,5,185,2020-03-26,2020-04-03
6,volatility,10,724,10,184,2020-03-13,2020-03-30
7,volatility,21,704,21,182,2020-02-13,2020-03-17
8,direction,1,739,1,186,2020-04-03,2020-04-07
9,direction,5,733,5,185,2020-03-26,2020-04-03


In [39]:
# Activate FINAL experiment architecture
# Preserve the earlier screening version for reference


experiments_screening = experiments
experiments = experiments_final

print("FINAL experiments activated.")
print("Number of experiments:", len(experiments))

print("\nPredictor counts for 21-day volatility:")
print(
    "Macro:",
    experiments[("volatility", 21)]["X_macro"].shape[1]
)
print(
    "DeepSeek:",
    experiments[("volatility", 21)]["X_ds"].shape[1]
)
print(
    "Gemma:",
    experiments[("volatility", 21)]["X_gm"].shape[1]
)

print("\nSplit sizes for 21-day volatility:")
print(
    "Train:",
    len(experiments[("volatility", 21)]["train_idx"])
)
print(
    "Purged:",
    len(experiments[("volatility", 21)]["purged_idx"])
)
print(
    "Test:",
    len(experiments[("volatility", 21)]["test_idx"])
)

FINAL experiments activated.
Number of experiments: 12

Predictor counts for 21-day volatility:
Macro: 43
DeepSeek: 130
Gemma: 130

Split sizes for 21-day volatility:
Train: 704
Purged: 21
Test: 182


In [40]:
# Final sanity check under the corrected predictor architecture

vol1_final_results, vol1_final_models, vol1_final_predictions = (
    run_regression_experiment(
        target_type="volatility",
        h=1
    )
)

vol1_final_table = pd.DataFrame(vol1_final_results).T

display(vol1_final_table)

print("\nSelected alphas:")
for model_name, model in vol1_final_models.items():
    print(
        f"{model_name}:",
        model.best_params_["lasso__alpha"]
    )

print("\nPredictor counts:")
print(
    "Macro:",
    experiments[("volatility", 1)]["X_macro"].shape[1]
)
print(
    "DeepSeek:",
    experiments[("volatility", 1)]["X_ds"].shape[1]
)
print(
    "Gemma:",
    experiments[("volatility", 1)]["X_gm"].shape[1]
)

print("\nSelected non-zero predictors:")

for model_name, model in vol1_final_models.items():

    if model_name == "Macro":
        feature_names = experiments[
            ("volatility", 1)
        ]["X_macro"].columns

    elif model_name == "DeepSeek":
        feature_names = experiments[
            ("volatility", 1)
        ]["X_ds"].columns

    else:
        feature_names = experiments[
            ("volatility", 1)
        ]["X_gm"].columns

    coefs = model.best_estimator_.named_steps[
        "lasso"
    ].coef_

    selected = pd.DataFrame({
        "predictor": feature_names,
        "coefficient": coefs
    })

    selected = selected[
        ~np.isclose(
            selected["coefficient"],
            0.0,
            atol=1e-12
        )
    ].copy()

    selected["abs_coefficient"] = (
        selected["coefficient"].abs()
    )

    selected = (
        selected
        .sort_values(
            "abs_coefficient",
            ascending=False
        )
        .drop(columns="abs_coefficient")
    )

    print(f"\n{model_name}: {len(selected)} selected")

    if len(selected) == 0:
        print("No predictors selected.")
    else:
        display(selected.reset_index(drop=True))

,RMSE,MAE,R2
Benchmark,0.002460,0.001953,-0.003842
Macro,0.002456,0.002019,-0.000861
DeepSeek,0.002451,0.002001,0.003282
Gemma,0.002474,0.002064,-0.016001



Selected alphas:
Macro: 0.0003409285069746811
DeepSeek: 0.000419870708444391
Gemma: 0.00029673024081888695

Predictor counts:
Macro: 43
DeepSeek: 130
Gemma: 130

Selected non-zero predictors:

Macro: 1 selected


,predictor,coefficient
0,VIXCLS,0.000295



DeepSeek: 2 selected


,predictor,coefficient
0,VIXCLS,0.000217
1,trade_sum_ma60,-0.000031



Gemma: 5 selected


,predictor,coefficient
0,VIXCLS,0.000336
1,trade_sum_ma60,-0.000070
2,trade_sum_ma30,-0.000038
3,sanctions_sum_ma30,-0.000026
4,fed_pressure_sum_ma60,-0.000005


In [ ]:
# DEFINITIVE target-horizon matrix
# 43 macro/EURUSD predictors
# 130 DeepSeek predictors
# 130 Gemma predictors

final_results = {}
final_models = {}
final_predictions = {}

# Return and volatility regression

for target_type in ["return", "volatility"]:
    for h in HORIZONS:

        print(f"Running FINAL {target_type} - {h}d...")

        results, models, predictions = (
            run_regression_experiment(
                target_type=target_type,
                h=h
            )
        )

        final_results[(target_type, h)] = results
        final_models[(target_type, h)] = models
        final_predictions[(target_type, h)] = predictions


# Direction classification

for h in HORIZONS:

    print(f"Running FINAL direction - {h}d...")

    results, models, predictions = (
        run_classification_experiment(h=h)
    )

    final_results[("direction", h)] = results
    final_models[("direction", h)] = models
    final_predictions[("direction", h)] = predictions


print("\nFINAL RUN COMPLETE.")
print(
    "Number of target-horizon experiments:",
    len(final_results)
)

Running FINAL return - 1d...
Running FINAL return - 5d...
Running FINAL return - 10d...
Running FINAL return - 21d...
Running FINAL volatility - 1d...
Running FINAL volatility - 5d...
Running FINAL volatility - 10d...
Running FINAL volatility - 21d...
Running FINAL direction - 1d...
Running FINAL direction - 5d...
Running FINAL direction - 10d...
Running FINAL direction - 21d...

FINAL RUN COMPLETE.
Number of target-horizon experiments: 12


In [ ]:
# SAVE DEFINITIVE TARGET-HORIZON RUN

OUTPUT_DIR = Path("../outputs/target_horizon_matrix")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save model-performance results

result_rows = []

for (target_type, h), results in final_results.items():

    for model_name, metrics in results.items():

        row = {
            "target": target_type,
            "horizon": h,
            "model": model_name
        }

        row.update(metrics)
        result_rows.append(row)

final_results_df = pd.DataFrame(result_rows)

final_results_df.to_csv(
    OUTPUT_DIR / "target_horizon_final_results.csv",
    index=False
)


# 2. Save test-set predictions

prediction_rows = []

for (target_type, h), preds in final_predictions.items():

    exp = experiments[(target_type, h)]
    test_idx = exp["test_idx"]

    temp = pd.DataFrame({
        "observation_date": exp["dates"].iloc[test_idx].values,
        "target": target_type,
        "horizon": h,
        "actual": exp["y"].iloc[test_idx].values
    })

    for model_name, pred_values in preds.items():
        temp[model_name] = pred_values

    prediction_rows.append(temp)

final_predictions_df = pd.concat(
    prediction_rows,
    ignore_index=True
)

final_predictions_df.to_csv(
    OUTPUT_DIR / "target_horizon_final_predictions.csv",
    index=False
)


print("Saved:")
print(OUTPUT_DIR / "target_horizon_final_results.csv")
print(OUTPUT_DIR / "target_horizon_final_predictions.csv")

print("\nNumber of experiments saved:", len(final_results))

Saved:
..\outputs\target_horizon_matrix\target_horizon_final_results.csv
..\outputs\target_horizon_matrix\target_horizon_final_predictions.csv

Number of experiments saved: 12


In [43]:
# FINAL 12-ROW TARGET-HORIZON PERFORMANCE MATRIX

matrix_rows = []

for target_type in ["return", "volatility", "direction"]:
    for h in HORIZONS:

        res = final_results[(target_type, h)]

        if target_type in ["return", "volatility"]:

            # Primary metric: RMSE (lower is better)
            benchmark = res["Benchmark"]["RMSE"]
            macro = res["Macro"]["RMSE"]
            ds = res["DeepSeek"]["RMSE"]
            gm = res["Gemma"]["RMSE"]

            ds_vs_macro = 100 * (macro - ds) / macro
            gm_vs_macro = 100 * (macro - gm) / macro

            values = {
                "Benchmark": benchmark,
                "Macro": macro,
                "DeepSeek": ds,
                "Gemma": gm
            }

            best_model = min(values, key=values.get)

            matrix_rows.append({
                "target": target_type,
                "horizon": h,
                "metric": "RMSE",
                "Benchmark": benchmark,
                "Macro": macro,
                "DeepSeek": ds,
                "Gemma": gm,
                "DeepSeek_vs_Macro": ds_vs_macro,
                "Gemma_vs_Macro": gm_vs_macro,
                "best_model": best_model
            })

        else:

            # Primary metric: ROC-AUC (higher is better)
            benchmark = res["Benchmark"]["ROC_AUC"]
            macro = res["Macro"]["ROC_AUC"]
            ds = res["DeepSeek"]["ROC_AUC"]
            gm = res["Gemma"]["ROC_AUC"]

            ds_vs_macro = ds - macro
            gm_vs_macro = gm - macro

            values = {
                "Benchmark": benchmark,
                "Macro": macro,
                "DeepSeek": ds,
                "Gemma": gm
            }

            best_model = max(values, key=values.get)

            matrix_rows.append({
                "target": target_type,
                "horizon": h,
                "metric": "ROC_AUC",
                "Benchmark": benchmark,
                "Macro": macro,
                "DeepSeek": ds,
                "Gemma": gm,
                "DeepSeek_vs_Macro": ds_vs_macro,
                "Gemma_vs_Macro": gm_vs_macro,
                "best_model": best_model
            })


final_target_matrix = pd.DataFrame(matrix_rows)

display(
    final_target_matrix.round(6)
)


# Save
final_target_matrix.to_csv(
    OUTPUT_DIR / "target_horizon_final_matrix.csv",
    index=False
)

print("\nSaved:")
print(OUTPUT_DIR / "target_horizon_final_matrix.csv")

,target,horizon,metric,Benchmark,Macro,DeepSeek,Gemma,DeepSeek_vs_Macro,Gemma_vs_Macro,best_model
0,return,1,RMSE,0.004069,0.004069,0.004069,0.004069,0.000000,0.000000,Benchmark
1,return,5,RMSE,0.009558,0.009558,0.009558,0.009558,0.000000,0.000000,Benchmark
2,return,10,RMSE,0.013518,0.013793,0.013793,0.013793,0.000000,0.000000,Benchmark
3,return,21,RMSE,0.021974,0.021974,0.021974,0.019980,0.000000,9.074197,Gemma
4,volatility,1,RMSE,0.002460,0.002456,0.002451,0.002474,0.207173,-0.753524,DeepSeek
5,volatility,5,RMSE,0.002738,0.003013,0.003531,0.003127,-17.190047,-3.778712,Benchmark
6,volatility,10,RMSE,0.002666,0.002501,0.004489,0.003453,-79.470854,-38.028174,Macro
7,volatility,21,RMSE,0.003684,0.003684,0.003584,0.003421,2.706947,7.144779,Gemma
8,direction,1,ROC_AUC,0.500000,0.511288,0.520997,0.523687,0.009709,0.012399,Gemma
9,direction,5,ROC_AUC,0.500000,0.648373,0.583549,0.606921,-0.064824,-0.041451,Macro



Saved:
..\outputs\target_horizon_matrix\target_horizon_final_matrix.csv


In [44]:
# Identify LLM models that outperform Macro

outperformance_rows = []

for _, row in final_target_matrix.iterrows():

    # For return/volatility these are % RMSE improvements.
    # For direction these are absolute ROC-AUC improvements.
    if row["DeepSeek_vs_Macro"] > 0:
        outperformance_rows.append({
            "target": row["target"],
            "horizon": row["horizon"],
            "LLM": "DeepSeek",
            "metric": row["metric"],
            "Macro": row["Macro"],
            "LLM_value": row["DeepSeek"],
            "improvement_vs_macro": row["DeepSeek_vs_Macro"]
        })

    if row["Gemma_vs_Macro"] > 0:
        outperformance_rows.append({
            "target": row["target"],
            "horizon": row["horizon"],
            "LLM": "Gemma",
            "metric": row["metric"],
            "Macro": row["Macro"],
            "LLM_value": row["Gemma"],
            "improvement_vs_macro": row["Gemma_vs_Macro"]
        })


llm_outperformance = pd.DataFrame(outperformance_rows)

if len(llm_outperformance) == 0:
    print("No DeepSeek or Gemma model beats the macro benchmark.")
else:
    llm_outperformance = (
        llm_outperformance
        .sort_values(
            ["target", "horizon", "improvement_vs_macro"],
            ascending=[True, True, False]
        )
        .reset_index(drop=True)
    )

    display(llm_outperformance.round(6))


# Save
llm_outperformance.to_csv(
    OUTPUT_DIR / "target_horizon_llm_outperformance.csv",
    index=False
)

,target,horizon,LLM,metric,Macro,LLM_value,improvement_vs_macro
0,direction,1,Gemma,ROC_AUC,0.511288,0.523687,0.012399
1,direction,1,DeepSeek,ROC_AUC,0.511288,0.520997,0.009709
2,return,21,Gemma,RMSE,0.021974,0.019980,9.074197
3,volatility,1,DeepSeek,RMSE,0.002456,0.002451,0.207173
4,volatility,21,Gemma,RMSE,0.003684,0.003421,7.144779
5,volatility,21,DeepSeek,RMSE,0.003684,0.003584,2.706947


In [ ]:
# FINAL SIGNIFICANCE / ROBUSTNESS TESTS
# Correct prediction keys for regression vs classification

import statsmodels.api as sm

# 1. HAC loss-differential test
#    Return / volatility

def hac_loss_test(y_true, pred_macro, pred_llm, maxlags):

    y_true = np.asarray(y_true)
    pred_macro = np.asarray(pred_macro)
    pred_llm = np.asarray(pred_llm)

    # Positive value means LLM has lower squared loss
    loss_macro = (y_true - pred_macro) ** 2
    loss_llm = (y_true - pred_llm) ** 2

    d = loss_macro - loss_llm

    fit = sm.OLS(
        d,
        np.ones(len(d))
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags}
    )

    return {
        "mean_loss_improvement": d.mean(),
        "test_stat": float(fit.tvalues[0]),
        "p_value": float(fit.pvalues[0])
    }


# 2. Moving-block bootstrap
#    Direction ROC-AUC

def moving_block_auc_test(
    y_true,
    prob_macro,
    prob_llm,
    block_length,
    n_boot=5000,
    random_state=42
):

    rng = np.random.default_rng(random_state)

    y_true = np.asarray(y_true).astype(int)
    prob_macro = np.asarray(prob_macro)
    prob_llm = np.asarray(prob_llm)

    n = len(y_true)

    observed_macro = roc_auc_score(
        y_true,
        prob_macro
    )

    observed_llm = roc_auc_score(
        y_true,
        prob_llm
    )

    observed_diff = (
        observed_llm - observed_macro
    )

    possible_starts = np.arange(
        0,
        n - block_length + 1
    )

    boot_diffs = []

    for _ in range(n_boot):

        indices = []

        while len(indices) < n:

            start = rng.choice(
                possible_starts
            )

            block = np.arange(
                start,
                start + block_length
            )

            indices.extend(
                block.tolist()
            )

        indices = np.asarray(
            indices[:n]
        )

        y_b = y_true[indices]

        if np.unique(y_b).size < 2:
            continue

        macro_auc = roc_auc_score(
            y_b,
            prob_macro[indices]
        )

        llm_auc = roc_auc_score(
            y_b,
            prob_llm[indices]
        )

        boot_diffs.append(
            llm_auc - macro_auc
        )

    boot_diffs = np.asarray(
        boot_diffs
    )

    ci_low, ci_high = np.percentile(
        boot_diffs,
        [2.5, 97.5]
    )

    return {
        "auc_improvement": observed_diff,
        "ci_2.5": ci_low,
        "ci_97.5": ci_high,
        "n_boot_valid": len(boot_diffs)
    }


# 3. Run tests only where an LLM beat Macro

significance_rows = []

for _, row in llm_outperformance.iterrows():

    target_type = row["target"]
    h = int(row["horizon"])
    llm_name = row["LLM"]

    preds = final_predictions[
        (target_type, h)
    ]

    # y_test was already stored during modelling
    y_test = np.asarray(
        preds["y_test"]
    )


    
    # RETURN / VOLATILITY

    if target_type in [
        "return",
        "volatility"
    ]:

        macro_pred = np.asarray(
            preds["Macro"]
        )

        llm_pred = np.asarray(
            preds[llm_name]
        )

        test = hac_loss_test(
            y_true=y_test,
            pred_macro=macro_pred,
            pred_llm=llm_pred,
            maxlags=h
        )

        significance_rows.append({
            "target": target_type,
            "horizon": h,
            "LLM": llm_name,
            "test": "HAC loss differential",
            "performance_improvement":
                row["improvement_vs_macro"],
            "loss_improvement":
                test["mean_loss_improvement"],
            "statistic":
                test["test_stat"],
            "CI_low": np.nan,
            "CI_high": np.nan,
            "p_value":
                test["p_value"]
        })


    # ========================================================
    # DIRECTION
    # ========================================================

    else:

        macro_prob = np.asarray(
            preds["Macro_prob"]
        )

        llm_prob = np.asarray(
            preds[f"{llm_name}_prob"]
        )

        test = moving_block_auc_test(
            y_true=y_test,
            prob_macro=macro_prob,
            prob_llm=llm_prob,
            block_length=max(h, 5),
            n_boot=5000,
            random_state=42
        )

        significance_rows.append({
            "target": target_type,
            "horizon": h,
            "LLM": llm_name,
            "test": "Moving-block AUC bootstrap",
            "performance_improvement":
                test["auc_improvement"],
            "loss_improvement": np.nan,
            "statistic": np.nan,
            "CI_low":
                test["ci_2.5"],
            "CI_high":
                test["ci_97.5"],
            "p_value": np.nan
        })


# Results

final_significance = pd.DataFrame(
    significance_rows
)

display(
    final_significance.round(6)
)


# Save
final_significance.to_csv(
    OUTPUT_DIR /
    "target_horizon_final_significance_tests.csv",
    index=False
)

print("\nSaved:")
print(
    OUTPUT_DIR /
    "target_horizon_final_significance_tests.csv"
)

,target,horizon,LLM,test,performance_improvement,loss_improvement,statistic,CI_low,CI_high,p_value
0,direction,1,Gemma,Moving-block AUC bootstrap,0.012399,NaN,NaN,-0.043399,0.071532,NaN
1,direction,1,DeepSeek,Moving-block AUC bootstrap,0.009709,NaN,NaN,-0.045695,0.072296,NaN
2,return,21,Gemma,HAC loss differential,9.074197,0.000084,2.212490,NaN,NaN,0.026933
3,volatility,1,DeepSeek,HAC loss differential,0.207173,0.000000,0.660232,NaN,NaN,0.509105
4,volatility,21,Gemma,HAC loss differential,7.144779,0.000002,1.664711,NaN,NaN,0.095970
5,volatility,21,DeepSeek,HAC loss differential,2.706947,0.000001,0.789429,NaN,NaN,0.429861



Saved:
..\outputs\target_horizon_matrix\target_horizon_final_significance_tests.csv


In [ ]:
# Inspect FINAL 21-day return models

target_type = "return"
h = 21

exp = experiments[(target_type, h)]
models = final_models[(target_type, h)]

for model_name in ["Macro", "DeepSeek", "Gemma"]:

    if model_name == "Macro":
        feature_names = exp["X_macro"].columns
    elif model_name == "DeepSeek":
        feature_names = exp["X_ds"].columns
    else:
        feature_names = exp["X_gm"].columns

    model = models[model_name]

    alpha = model.best_params_["lasso__alpha"]

    coefs = model.best_estimator_.named_steps[
        "lasso"
    ].coef_

    selected = pd.DataFrame({
        "predictor": feature_names,
        "coefficient": coefs
    })

    selected = selected[
        ~np.isclose(
            selected["coefficient"],
            0.0,
            atol=1e-12
        )
    ].copy()

    selected["abs_coefficient"] = (
        selected["coefficient"].abs()
    )

    selected = (
        selected
        .sort_values(
            "abs_coefficient",
            ascending=False
        )
        .drop(columns="abs_coefficient")
        .reset_index(drop=True)
    )

    print(f"\n{model_name}")
    print(f"Selected alpha: {alpha}")
    print(f"Non-zero predictors: {len(selected)}")

    if len(selected) == 0:
        print("No predictors selected.")
    else:
        display(selected)


Macro
Selected alpha: 0.007752597488629456
Non-zero predictors: 0
No predictors selected.

DeepSeek
Selected alpha: 0.008309941949353387
Non-zero predictors: 0
No predictors selected.

Gemma
Selected alpha: 0.005111433483440166
Non-zero predictors: 2


,predictor,coefficient
0,DGS2,-0.001100
1,trade_sum_ma60,-0.000218


In [ ]:
# FULL REGRESSION INFERENCE MATRIX
# DeepSeek/Gemma vs Macro for every return/volatility horizon
# + multiple-testing corrections

from statsmodels.stats.multitest import multipletests

full_regression_tests = []

for target_type in ["return", "volatility"]:
    for h in HORIZONS:

        preds = final_predictions[(target_type, h)]

        y_test = np.asarray(preds["y_test"])
        macro_pred = np.asarray(preds["Macro"])

        for llm_name in ["DeepSeek", "Gemma"]:

            llm_pred = np.asarray(
                preds[llm_name]
            )

            test = hac_loss_test(
                y_true=y_test,
                pred_macro=macro_pred,
                pred_llm=llm_pred,
                maxlags=h
            )

            macro_rmse = final_results[
                (target_type, h)
            ]["Macro"]["RMSE"]

            llm_rmse = final_results[
                (target_type, h)
            ][llm_name]["RMSE"]

            rmse_improvement = (
                100
                * (macro_rmse - llm_rmse)
                / macro_rmse
            )

            full_regression_tests.append({
                "target": target_type,
                "horizon": h,
                "LLM": llm_name,
                "Macro_RMSE": macro_rmse,
                "LLM_RMSE": llm_rmse,
                "RMSE_improvement_pct": rmse_improvement,
                "loss_improvement":
                    test["mean_loss_improvement"],
                "HAC_statistic":
                    test["test_stat"],
                "p_raw":
                    test["p_value"]
            })


full_regression_inference = pd.DataFrame(
    full_regression_tests
)


# Multiple-testing corrections

pvals = full_regression_inference[
    "p_raw"
].to_numpy()

# Benjamini-Hochberg:
# controls false discovery rate
_, p_fdr, _, _ = multipletests(
    pvals,
    alpha=0.05,
    method="fdr_bh"
)

# Holm:
# stricter family-wise error correction
_, p_holm, _, _ = multipletests(
    pvals,
    alpha=0.05,
    method="holm"
)

full_regression_inference[
    "p_FDR_BH"
] = p_fdr

full_regression_inference[
    "p_Holm"
] = p_holm


# Sort by raw p-value
full_regression_inference = (
    full_regression_inference
    .sort_values("p_raw")
    .reset_index(drop=True)
)

display(
    full_regression_inference.round(6)
)


# Save
full_regression_inference.to_csv(
    OUTPUT_DIR /
    "target_horizon_full_regression_inference.csv",
    index=False
)

,target,horizon,LLM,Macro_RMSE,LLM_RMSE,RMSE_improvement_pct,loss_improvement,HAC_statistic,p_raw,p_FDR_BH,p_Holm
0,volatility,5,DeepSeek,0.003013,0.003531,-17.190047,-0.000003,-5.166640,0.000000,NaN,0.000004
1,volatility,10,DeepSeek,0.002501,0.004489,-79.470854,-0.000014,-4.798485,0.000002,NaN,0.000024
2,volatility,5,Gemma,0.003013,0.003127,-3.778712,-0.000001,-3.831139,0.000128,NaN,0.001786
3,volatility,10,Gemma,0.002501,0.003453,-38.028174,-0.000006,-3.501201,0.000463,NaN,0.006021
4,return,21,Gemma,0.021974,0.019980,9.074197,0.000084,2.212490,0.026933,NaN,0.323194
5,volatility,21,Gemma,0.003684,0.003421,7.144779,0.000002,1.664711,0.095970,NaN,1.000000
6,volatility,1,Gemma,0.002456,0.002474,-0.753524,-0.000000,-1.563581,0.117916,NaN,1.000000
7,volatility,21,DeepSeek,0.003684,0.003584,2.706947,0.000001,0.789429,0.429861,NaN,1.000000
8,volatility,1,DeepSeek,0.002456,0.002451,0.207173,0.000000,0.660232,0.509105,NaN,1.000000
9,return,1,DeepSeek,0.004069,0.004069,0.000000,0.000000,NaN,NaN,NaN,NaN


In [ ]:
# FIX MULTIPLE-TESTING CORRECTIONS
# Exact-identical forecasts -> p = 1
# Correct across all 16 pre-specified regression comparisons

full_regression_inference_corrected = (
    full_regression_inference.copy()
)

# If predictions/losses are exactly identical, HAC is undefined
# because the loss differential has zero variance.
# Treat these comparisons as p = 1 (no evidence of difference).
full_regression_inference_corrected["p_for_correction"] = (
    full_regression_inference_corrected["p_raw"]
    .fillna(1.0)
)

pvals = (
    full_regression_inference_corrected[
        "p_for_correction"
    ]
    .to_numpy()
)

# Benjamini-Hochberg FDR
_, p_fdr, _, _ = multipletests(
    pvals,
    alpha=0.05,
    method="fdr_bh"
)

# Holm family-wise correction
_, p_holm, _, _ = multipletests(
    pvals,
    alpha=0.05,
    method="holm"
)

full_regression_inference_corrected[
    "p_FDR_BH"
] = p_fdr

full_regression_inference_corrected[
    "p_Holm"
] = p_holm

# Keep original raw p-values visible;
# exact-null models remain NaN there because HAC itself is undefined.
full_regression_inference_corrected = (
    full_regression_inference_corrected
    .sort_values(
        "p_for_correction"
    )
    .reset_index(drop=True)
)

display(
    full_regression_inference_corrected[
        [
            "target",
            "horizon",
            "LLM",
            "RMSE_improvement_pct",
            "HAC_statistic",
            "p_raw",
            "p_FDR_BH",
            "p_Holm"
        ]
    ].round(6)
)

full_regression_inference_corrected.to_csv(
    OUTPUT_DIR /
    "target_horizon_full_regression_inference_corrected.csv",
    index=False
)


,target,horizon,LLM,RMSE_improvement_pct,HAC_statistic,p_raw,p_FDR_BH,p_Holm
0,volatility,5,DeepSeek,-17.190047,-5.166640,0.000000,0.000004,0.000004
1,volatility,10,DeepSeek,-79.470854,-4.798485,0.000002,0.000013,0.000024
2,volatility,5,Gemma,-3.778712,-3.831139,0.000128,0.000680,0.001786
3,volatility,10,Gemma,-38.028174,-3.501201,0.000463,0.001853,0.006021
4,return,21,Gemma,9.074197,2.212490,0.026933,0.086185,0.323194
5,volatility,21,Gemma,7.144779,1.664711,0.095970,0.255921,1.000000
6,volatility,1,Gemma,-0.753524,-1.563581,0.117916,0.269523,1.000000
7,volatility,21,DeepSeek,2.706947,0.789429,0.429861,0.859723,1.000000
8,volatility,1,DeepSeek,0.207173,0.660232,0.509105,0.905076,1.000000
9,return,1,DeepSeek,0.000000,NaN,NaN,1.000000,1.000000
